# SpaceX Falcon 9 — EDA with SQL

Loads the wrangled launch dataset into a SQLite table `SPACEXTABLE` and answers standard analysis questions with SQL.

In [ ]:
import sqlite3
import pandas as pd
import json

d = pd.read_csv("data/dataset_part_2.csv")

conn = sqlite3.connect(":memory:")
d.to_sql("SPACEXTABLE", conn, index=False, if_exists="replace")

queries = {
    "Q1: Distinct launch sites": "SELECT DISTINCT LaunchSite FROM SPACEXTABLE;",
    "Q2: Sample launch sites starting with 'CCA'": "SELECT * FROM SPACEXTABLE WHERE LaunchSite LIKE 'CCA%' LIMIT 5;",
    "Q3: Total payload mass carried by boosters launched to ISS orbit (NASA CRS resupply missions)": "SELECT SUM(PayloadMass) AS total_payload_mass_kg FROM SPACEXTABLE WHERE Orbit = 'ISS';",
    "Q4: Average payload mass per orbit type": "SELECT Orbit, ROUND(AVG(PayloadMass),1) AS avg_payload_mass_kg FROM SPACEXTABLE GROUP BY Orbit ORDER BY avg_payload_mass_kg DESC;",
    "Q5: Date of first successful ground-pad landing": "SELECT MIN(Date) AS first_success_ground_pad FROM SPACEXTABLE WHERE Outcome = 'True RTLS';",
    "Q6: Boosters with successful drone-ship landing and payload 4000-6000kg": "SELECT DISTINCT Serial FROM SPACEXTABLE WHERE Outcome = 'True ASDS' AND PayloadMass BETWEEN 4000 AND 6000;",
    "Q7: Total successful vs failed mission outcomes": "SELECT Class, COUNT(*) AS n FROM SPACEXTABLE GROUP BY Class;",
    "Q8: Booster(s) that carried the maximum payload mass": "SELECT Serial, PayloadMass FROM SPACEXTABLE WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTABLE);",
    "Q9: Records with drone-ship landing failure in 2015 (by month)": "SELECT strftime('%m', Date) AS month, Serial, LaunchSite FROM SPACEXTABLE WHERE Outcome = 'False ASDS' AND strftime('%Y', Date) = '2015';",
    "Q10: Landing outcome counts between 2010-06-04 and 2017-03-20, ranked": "SELECT Outcome, COUNT(*) AS n FROM SPACEXTABLE WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Outcome ORDER BY n DESC;",
}

results = {}
for label, q in queries.items():
    cur = conn.execute(q)
    cols = [c[0] for c in cur.description]
    rows = cur.fetchall()
    results[label] = {"sql": q, "columns": cols, "rows": rows}
    print("="*80)
    print(label)
    print(q)
    print(cols)
    for r in rows[:10]:
        print(r)

with open("sql_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)


## Query Results Summary

1. Distinct launch sites: CCAFS SLC 40, VAFB SLC 4E, KSC LC 39A
2. Sample CCAFS launches: 5 rows shown
3. Total payload to ISS orbit: 68,878.7 kg
4. Average payload by orbit: highest VLEO (15,315.7 kg), lowest ES-L1 (570.0 kg)
5. First successful ground-pad landing: 2015-12-22
6. Boosters with drone-ship landing + 4000-6000kg payload: B1022, B1026, B1021, B1031, B1046, B1059
7. Outcome counts: 30 failed landings (Class 0), 60 successful (Class 1)
8. Max payload (15,600 kg): boosters B1048, B1051
9. Drone-ship failures in 2015: January (B1012) and April (B1015), both CCAFS SLC 40
10. Outcome counts 2010-06-04 to 2017-03-20: None None (9), True ASDS (5), False ASDS (4), True RTLS (3), True Ocean (3)